In [0]:
import json
import time
import pyspark.sql.functions as sf
from functools import reduce

CHECKPOINT_PATH = "/Volumes/dev/raw/checkpoints/azure-trip-updates_checkpoint_batch.json"
DATA_PATH = "/Volumes/dev/raw/ttc_trip_updates_volume"

In [0]:
def get_sorted_files(path: list[str]) -> list[str]:
    """Return .jsonl files sorted oldest -> newest"""
    files = [f.name for f in dbutils.fs.ls(path) if f.name.endswith(".jsonl")]
    files.sort(key=lambda f: time.strptime(f.split("_")[2].replace(".jsonl", ""), "%Y%m%dT%H"))
    return files

def load_checkpoint() -> dict:
    try:
        with open(CHECKPOINT_PATH, "r") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {"last_file": None}

def save_checkpoint(last_file: str) -> None:
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump({"last_file": last_file}, f)


def read_file(file: str):
    """Read rows return (df)."""
    df = spark.read.json(f"{DATA_PATH}/{file}")
    df = df.withColumn("source_name", sf.lit(file))
    df = df.withColumn("ingested_at", sf.lit(int(time.time())))
    # Extract date hour from file name, e.g., trip_updates_20260305T00
    date_hour = file.split("_")[2].replace(".jsonl", "")
    df = df.withColumn("date_hour", sf.lit(date_hour))
    return df

In [0]:
def batch() -> list:
    """
    Read all unprocessed files in chronological order.

    - Files older than the checkpointed file are skipped entirely.
    - All newer files are read.
    """
    checkpoint = load_checkpoint()
    last_file = checkpoint["last_file"]

    all_files = get_sorted_files(DATA_PATH)  # oldest -> newest

    # Determine where to start processing
    if last_file is None:
        # No checkpoint: process everything from the beginning
        start_idx = 0
    elif last_file in all_files:
        start_idx = all_files.index(last_file)
    else:
        # Checkpointed file no longer exists; start fresh
        start_idx = 0

    fetched_data = []

    for i, file in enumerate(all_files[start_idx:], start=start_idx):
        df = read_file(file)
        if df is not None:  # skip empty/fully-filtered files
            fetched_data.append(df)
            save_checkpoint(file)

    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), fetched_data) if fetched_data else None

In [0]:
df = batch()
if df:
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    df.write.format("delta").mode("overwrite").partitionBy("date_hour").saveAsTable("dev.raw.trip_updates_batch")

In [0]:
display(df.count())